In [12]:
# ============================================================================
# SECTION 1: IMPORTS - Core Libraries for Acuity Prediction Pipeline
# ============================================================================
# This notebook implements a single-stage acuity prediction pipeline:
# INPUT: Clinical data (vitals, demographics, chief complaints)
# OUTPUT: Predicted acuity level (ESI 1-5)
#
# Libraries imported:
# - Data manipulation: pandas, numpy
# - ML & preprocessing: scikit-learn pipelines, transformers, model selection
# - Gradient boosting: LightGBM (primary model for acuity classification)
# - NLP: HuggingFace Transformers (ClinicalBERT for clinical text embeddings)
# - Visualization: matplotlib, seaborn
# - Text processing: TfidfVectorizer, chi2 feature selection
# - Model evaluation: confusion matrix, f1-score, cohen_kappa_score (ordinal metrics)

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn: preprocessing, pipelines, and model selection
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, 
    cohen_kappa_score, 
    confusion_matrix, 
    classification_report,
    f1_score,
    roc_auc_score
)

# Ensemble and gradient boosting models
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from lightgbm import LGBMClassifier

# Text processing for chief complaint narratives
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import chi2
from sklearn.base import BaseEstimator, TransformerMixin

# Transformers library: state-of-the-art NLP models for clinical text
try:
    import torch
    from transformers import AutoTokenizer, AutoModel
    TRANSFORMERS_AVAILABLE = True
    DEVICE = torch.device('xpu' if torch.xpu.is_available() else 'cpu')
except ImportError:
    print("⚠️  HuggingFace transformers not installed. Will use TF-IDF fallback.")
    TRANSFORMERS_AVAILABLE = False
    DEVICE = 'cpu'

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seeds for reproducibility across runs
np.random.seed(42)

print("✓ All imports successful!")
print(f"✓ Transformers available: {TRANSFORMERS_AVAILABLE}")
print(f"✓ Device for embeddings: {DEVICE}")

✓ All imports successful!
✓ Transformers available: True
✓ Device for embeddings: xpu


In [13]:
# Load final MIMIC dataset with comorbidities and age_group
full_df = pd.read_csv("dataset/final_mimic.csv")
print(f"✓ Loaded final_mimic.csv")
print(f"  Shape: {full_df.shape}")
print(f"  Columns: {len(full_df.columns)}")
full_df.head()


✓ Loaded final_mimic.csv
  Shape: (251886, 60)
  Columns: 60


,patient_id,disposition,arrival_year,arrival_month,arrival_week,arrival_day,arrival_hour,arrival_season,shift,arrival_mode,...,hx_hyperthyroidism,hx_hiv,hx_coagulopathy,hx_immunosuppressed,hx_pregnant,hx_substance_use_disorder,hx_coronary_artery_disease,hx_stroke_prior,hx_peripheral_vascular_disease,age_group
0,13238787,eloped,2110,1,2,Saturday,1,winter,night,unknown,...,0,0,0,0,0,0,0,0,0,young_adult
1,15350437,admitted,2110,1,2,Saturday,3,winter,night,ambulance,...,0,0,0,0,0,1,0,0,0,young_adult
2,10871444,home,2110,1,3,Sunday,5,winter,night,walk-in,...,0,0,0,0,0,0,0,0,0,young_adult
3,19925345,admitted,2110,1,5,Thursday,7,winter,morning,ambulance,...,0,0,0,0,0,0,0,0,0,young_adult
4,10101340,home,2110,1,3,Saturday,9,winter,morning,walk-in,...,0,0,0,0,0,0,0,0,0,middle_aged


In [14]:
# Remove patient_id as it's not a feature for the model
# (patient_id is only for reference, not for prediction)
if 'patient_id' in full_df.columns:
    full_df = full_df.drop(columns=['patient_id'])

print(f"✓ Data cleaned")
print(f"  Columns after cleanup: {len(full_df.columns)}")
print(f"  Target variable 'triage_acuity' present: {'triage_acuity' in full_df.columns}")
full_df.head()


✓ Data cleaned
  Columns after cleanup: 59
  Target variable 'triage_acuity' present: True


,disposition,arrival_year,arrival_month,arrival_week,arrival_day,arrival_hour,arrival_season,shift,arrival_mode,sex,...,hx_hyperthyroidism,hx_hiv,hx_coagulopathy,hx_immunosuppressed,hx_pregnant,hx_substance_use_disorder,hx_coronary_artery_disease,hx_stroke_prior,hx_peripheral_vascular_disease,age_group
0,eloped,2110,1,2,Saturday,1,winter,night,unknown,male,...,0,0,0,0,0,0,0,0,0,young_adult
1,admitted,2110,1,2,Saturday,3,winter,night,ambulance,male,...,0,0,0,0,0,1,0,0,0,young_adult
2,home,2110,1,3,Sunday,5,winter,night,walk-in,female,...,0,0,0,0,0,0,0,0,0,young_adult
3,admitted,2110,1,5,Thursday,7,winter,morning,ambulance,female,...,0,0,0,0,0,0,0,0,0,young_adult
4,home,2110,1,3,Saturday,9,winter,morning,walk-in,male,...,0,0,0,0,0,0,0,0,0,middle_aged


In [15]:
class Chi2TextFeatureSelector(BaseEstimator, TransformerMixin):
    """
    Extracts and selects the most discriminative clinical keywords from medical text
    using TF-IDF vectorization + Chi-Square statistical feature selection.
    
    This is particularly effective for acuity triage where specific clinical terms
    are strong predictors of severity:
      Example: "respiratory distress", "shock", "altered mental status" → high acuity
      Example: "routine follow-up", "wellness check", "control visit" → low acuity
    
    The Chi-Square test identifies which TF-IDF features are most associated with
    each acuity level using a One-vs-Rest strategy.
    
    Parameters
    ----------
    text_col : str
        Name of the text column to process (default: 'chief_complaint_raw')
    k_per_label : int
        Number of top keywords to select for each acuity class (default: 50)
    min_df : int
        Minimum document frequency - ignores words in <min_df documents (default: 3)
    ngram_range : tuple
        N-gram range: (1,2) = unigrams + bigrams; (1,3) = unigrams + bigrams + trigrams
    
    Attributes
    ----------
    tfidf_vectorizer_ : TfidfVectorizer
        Fitted TF-IDF vectorizer (learned vocabulary)
    selected_indices_ : list
        Column indices in TF-IDF vocabulary selected by Chi-Square test
    feature_names_ : list
        Human-readable names for selected features
        Format: 'tfidf_word_c1_c2' (word predicts classes 1 and 2)
    """
    
    def __init__(self, 
                 text_col: str = 'chief_complaint_raw', 
                 k_per_label: int = 50, 
                 min_df: int = 3,  
                 ngram_range: tuple = (1, 3)):
        self.text_col = text_col
        self.k_per_label = k_per_label
        self.min_df = min_df
        self.ngram_range = ngram_range
        
        # Will be set during fit()
        self.tfidf_vectorizer_ = None
        self.selected_indices_ = None 
        self.feature_names_ = None
        
    def fit(self, X: pd.DataFrame, y) -> 'Chi2TextFeatureSelector':
        """
        Learn the TF-IDF vocabulary and identify top discriminative features per class.
        
        IMPORTANT: Requires target variable 'y' to compute Chi-Square statistics!
        When used in sklearn.compose.ColumnTransformer, call fit_transform(X, y).
        
        Parameters
        ----------
        X : pd.DataFrame
            Feature matrix containing the text column
        y : array-like
            Target variable (acuity labels 1-5 or 0-4)
        """
        if y is None:
            raise ValueError(
                "Target variable 'y' REQUIRED for Chi-Square feature selection!\n"
                "When using ColumnTransformer, call: preprocessor.fit_transform(X_train, y_train)"
            )
        # Ensure X is DataFrame
        if isinstance(X, np.ndarray):
            if X.ndim == 1:
                X = X.reshape(-1, 1)
            X = pd.DataFrame(X, columns=[self.text_col])
        
        # Convert y to numpy array
        y_arr = y.values if isinstance(y, pd.Series) else np.array(y)
        
        # Extract text column and fill NaN with empty string
        text_data = X[self.text_col].fillna('').astype(str)
        
        # ====================================================================
        # STEP 1: Fit TF-IDF Vectorizer with Snowball Stemming
        # ====================================================================
        print(f"📝 Fitting TF-IDF vectorizer with Snowball Stemmer...")
        print(f"   - min_df={self.min_df}, "
              f"ngrams={self.ngram_range}, "
              f"stopwords=English, "
              f"stemmer=Snowball")
        
        # Snowball Stemmer for medical term normalization
        from nltk.stem.snowball import SnowballStemmer
        stemmer = SnowballStemmer('english')
        
        # Custom tokenizer: split → lowercase → stem
        def tokenize_and_stem(text):
            tokens = text.lower().split()
            return [stemmer.stem(token) for token in tokens]
        
        self.tfidf_vectorizer_ = TfidfVectorizer(
            input='content',
            encoding='utf-8',
            lowercase=True,
            stop_words='english',  # Remove common English stopwords
            min_df=self.min_df,  # Ignore rare terms
            ngram_range=self.ngram_range,  # Extract unigrams and bigrams
            tokenizer=tokenize_and_stem  # Apply Snowball stemming to each token
        )
        
        # Compute TF-IDF scores: shape = (n_samples, n_features_tfidf)
        X_tfidf = self.tfidf_vectorizer_.fit_transform(text_data)
        print(f"   ✓ TF-IDF vocabulary size: {X_tfidf.shape[1]} terms")
        
        # ====================================================================
        # STEP 2: Chi-Square Feature Selection (One-vs-Rest)
        # ====================================================================
        # For each class, compute Chi2 scores between TF-IDF features and
        # a binary target (1 = class, 0 = not this class). Select top k features per class.
        
        print(f"📊 Selecting top {self.k_per_label} features per class via Chi-Square test...")
        unique_classes = np.unique(y_arr)
        print(f"   Classes: {unique_classes}")
        
        feature_to_labels = {}  # Track which classes each feature helps predict
        self.feature_to_chi2_scores_ = {}  # Track Chi2 scores per class per feature
        
        for label in unique_classes:
            # Create binary target: 1 if sample belongs to this class, 0 otherwise
            y_binary = (y_arr == label).astype(int)
            
            # Compute Chi2 scores between each TF-IDF feature and binary target
            # chi2() returns (scores, p_values) — we only use scores
            chi2_scores, _ = chi2(X_tfidf, y_binary)
            
            # How many features can we select? (Don't exceed vocabulary size)
            n_features = X_tfidf.shape[1]
            k_safe = min(self.k_per_label, n_features)
            
            if k_safe > 0:
                # Get indices of top k features (highest Chi2 scores)
                top_k_indices = np.argsort(chi2_scores)[-k_safe:]
                
                # Record that these feature indices are predictive for this class
                for idx in top_k_indices:
                    if idx not in feature_to_labels:
                        feature_to_labels[idx] = {}
                        self.feature_to_chi2_scores_[idx] = {}
                    feature_to_labels[idx][label] = chi2_scores[idx]
                    self.feature_to_chi2_scores_[idx][label] = chi2_scores[idx]
        
        # ====================================================================
        # STEP 3: Finalize Selected Indices and Create Feature Names
        # ====================================================================
        # Combine all indices selected across all classes (union)
        self.selected_indices_ = sorted(list(feature_to_labels.keys()))
        
        if not self.selected_indices_:
            print("⚠️  Warning: No features were selected!")
            self.feature_names_ = []
            return self
        
        # Get the actual words/n-grams from the TF-IDF vocabulary
        raw_feature_names = self.tfidf_vectorizer_.get_feature_names_out()
        self.feature_names_ = []
        
        for idx in self.selected_indices_:
            # Get the word/n-gram
            word = raw_feature_names[idx]
            
            # Get Chi2 scores for this feature across all classes
            chi2_per_class = feature_to_labels[idx]
            
            # Find the dominant class (highest Chi2 score)
            dominant_class = max(chi2_per_class, key=chi2_per_class.get)
            
            # Get all classes that predicted this feature, sorted by Chi2 score (descending)
            classes_sorted = sorted(chi2_per_class.keys(), 
                                   key=lambda x: chi2_per_class[x], 
                                   reverse=True)
            
            # Create feature name: dominant class first, then other classes
            # Format: tfidf_word_c0_2_3 where c0 is the dominant class
            classes_suffix = "_".join([f"c{lbl}" for lbl in classes_sorted])
            
            # Final feature name: e.g., "tfidf_apnea_c1_2"
            feature_name = f"tfidf_{word}_{classes_suffix}"
            self.feature_names_.append(feature_name)
        
        print(f"   ✓ Total unique features selected (union): {len(self.selected_indices_)}")
        return self
    
    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """
        Apply learned TF-IDF transformation and filter to selected features.
        """
        if self.tfidf_vectorizer_ is None:
            raise RuntimeError(
                "This transformer has not been fitted yet!\n"
                "Call fit() or fit_transform() first."
            )
        
        # Ensure X is DataFrame
        if isinstance(X, np.ndarray):
            if X.ndim == 1:
                X = X.reshape(-1, 1)
            X = pd.DataFrame(X, columns=[self.text_col])
        
        # Copy to avoid modifying original
        df = X.copy()
        text_data = df[self.text_col].fillna('').astype(str)
        
        # Remove raw text column (we don't want it in the final feature set)
        dfs_to_concat = [df.drop(columns=[self.text_col], errors='ignore')]
        
        if len(self.selected_indices_) > 0:
            # Transform using the full TF-IDF vocabulary learned during fit()
            X_tfidf_full = self.tfidf_vectorizer_.transform(text_data)
            
            # Select only the columns (features) that were chosen during fit()
            X_tfidf_selected = X_tfidf_full[:, self.selected_indices_]
            
            # Convert sparse matrix to dense DataFrame
            df_tfidf = pd.DataFrame(
                X_tfidf_selected.toarray(),  # Convert sparse to dense
                columns=self.feature_names_,  # Use descriptive feature names
                index=df.index  # Preserve original index for alignment
            )
            
            # Add TF-IDF features to concatenation list
            dfs_to_concat.append(df_tfidf)
        
        # Horizontally concatenate: other features + TF-IDF features
        final_df = pd.concat(dfs_to_concat, axis=1)
        return final_df

In [16]:
# ====================================================================
# ClinicalBERT Embedder for Medical Text Analysis
# ====================================================================
class ClinicalBERTEmbedder(BaseEstimator, TransformerMixin):
    """
    Extracts semantic embeddings from clinical text using ClinicalBERT.
    
    ClinicalBERT is trained on MIMIC-III clinical notes (87K+ records).
    It understands medical terminology, abbreviations, and clinical patterns
    better than generic BERT models.
    
    Uses XPU (Intel GPU) if available for faster inference, falls back to CPU.
    
    Features:
    - **Caching**: Saves embeddings to disk to avoid recomputation
    - Cache location: <cache_dir>/embeddings_<hash>.npy
    - First run: Computes embeddings and caches them
    - Subsequent runs: Loads from cache (instant!)
    
    Parameters
    ----------
    text_col : str
        Name of the text column to process (default: 'chief_complaint_raw')
    max_length : int
        Maximum token sequence length (default: 128)
    batch_size : int
        Number of samples to process at once (default: 16)
    cache_dir : str
        Directory to store cached embeddings (default: 'bert_cache')
        Create directory if it doesn't exist
    """
    
    def __init__(self,
                 text_col: str = 'chief_complaint_raw',
                 max_length: int = 128,
                 batch_size: int = 16,
                 cache_dir: str = None):
        if not TRANSFORMERS_AVAILABLE:
            raise RuntimeError(
                "HuggingFace transformers library is required.\n"
                "Install with: pip install torch transformers"
            )
        
        self.text_col = text_col
        self.max_length = max_length
        self.batch_size = batch_size
        self.cache_dir = cache_dir or 'bert_cache'
        self.model_ = None
        self.tokenizer_ = None
        self.device = DEVICE  # Use global DEVICE (XPU if available, else CPU)
        
        # Ensure cache directory exists
        import os
        os.makedirs(self.cache_dir, exist_ok=True)
    
    def fit(self, X, y=None):
        """Load ClinicalBERT model and tokenizer"""
        print(f"🏥 Loading ClinicalBERT (medical-domain embeddings)...")
        print(f"   Device: {self.device} (XPU optimized)" if 'xpu' in str(self.device) else f"   Device: {self.device}")
        print(f"   Cache directory: {self.cache_dir}")
        
        try:
            # Load ClinicalBERT from HuggingFace
            self.tokenizer_ = AutoTokenizer.from_pretrained('medicalai/ClinicalBERT')
            self.model_ = AutoModel.from_pretrained('medicalai/ClinicalBERT').to(self.device)
            self.model_.eval()
            print(f"   ✓ ClinicalBERT loaded (768-dim embeddings, MIMIC-III trained)")
        except Exception as e:
            print(f"   ⚠️  ClinicalBERT not available: {str(e)[:50]}...")
            print(f"   Falling back to generic BERT...")
            self.tokenizer_ = AutoTokenizer.from_pretrained('distilbert-base-uncased')
            self.model_ = AutoModel.from_pretrained('distilbert-base-uncased').to(self.device)
            self.model_.eval()
            print(f"   ✓ DistilBERT loaded (generic, fast, device={self.device})")
        
        return self
    
    def transform(self, X):
        """Extract [CLS] token embeddings with intelligent caching"""
        import hashlib
        import os
        
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X, columns=[self.text_col] if X.shape[1] == 1 else None)
        
        texts = X[self.text_col].fillna('').astype(str).tolist()
        
        # Create cache key based on text content and parameters
        # This ensures different datasets get different cache files
        text_content = '|'.join(texts)
        cache_key = hashlib.md5(
            f"{text_content}_{self.max_length}_{self.batch_size}".encode()
        ).hexdigest()
        cache_file = os.path.join(self.cache_dir, f"embeddings_{cache_key}.npy")
        
        # Try to load from cache first
        if os.path.exists(cache_file):
            print(f"💾 Loading embeddings from cache...")
            embeddings_array = np.load(cache_file)
            cache_size_mb = os.path.getsize(cache_file) / 1024 / 1024
            print(f"   ✓ Loaded {len(texts)} embeddings from cache ({cache_size_mb:.1f} MB)")
            print(f"   ✓ Embeddings shape: {embeddings_array.shape}")
            return embeddings_array
        
        # If not in cache, compute embeddings
        embeddings = []
        
        print(f"🏥 Computing embeddings for {len(texts)} texts (first time, will cache)...")
        print(f"   Device: {self.device}")
        
        self.model_.eval()
        with torch.no_grad():
            for i in range(0, len(texts), self.batch_size):
                batch_texts = texts[i:i + self.batch_size]
                
                encoded = self.tokenizer_(
                    batch_texts,
                    max_length=self.max_length,
                    padding='max_length',
                    truncation=True,
                    return_tensors='pt'
                )
                
                input_ids = encoded['input_ids'].to(self.device)
                attention_mask = encoded['attention_mask'].to(self.device)
                
                outputs = self.model_(input_ids, attention_mask=attention_mask)
                cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
                embeddings.append(cls_embeddings)
                
                # Progress indicator
                processed = min(i + self.batch_size, len(texts))
                if processed % (self.batch_size * 5) == 0 or processed == len(texts):
                    print(f"   Progress: {processed}/{len(texts)}")
        
        embeddings_array = np.vstack(embeddings)
        
        # Save to cache
        try:
            np.save(cache_file, embeddings_array)
            cache_size_mb = os.path.getsize(cache_file) / 1024 / 1024
            print(f"   ✓ Embeddings cached successfully ({cache_size_mb:.1f} MB)")
            print(f"   ✓ Cache file: {cache_file}")
        except Exception as e:
            print(f"   ⚠️  Could not save cache: {str(e)}")
        
        print(f"   ✓ Embeddings shape: {embeddings_array.shape}")
        
        return embeddings_array


print("✓ ClinicalBERTEmbedder class defined successfully (with caching)!")

✓ ClinicalBERTEmbedder class defined successfully (with caching)!


In [17]:
# ============================================================================
# Function: Advanced Feature Engineering
# ============================================================================

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create derived features to capture complex medical relationships.
    
    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe with raw features
    
    Returns
    -------
    pd.DataFrame
        Dataframe with new engineered features added
    """
    df = df.copy()  # Avoid modifying original
    
    print("🔧 ENGINEERING FEATURES...")
    
    # ====================================================================
    # A. BLOOD PRESSURE-DERIVED FEATURES
    # ====================================================================
    # These capture vascular and hemodynamic status not obvious from raw vitals
    
    if 'pulse_pressure' in df.columns and 'mean_arterial_pressure' in df.columns:
        # Pulse pressure ratio (avoid division by zero)
        df['pulse_pressure_ratio'] = df['pulse_pressure'] / (df['mean_arterial_pressure'] + 0.1)
        print("   ✓ pulse_pressure_ratio = pulse_pressure / (MAP + 0.1)")
    else:
        print("   ⚠️  Skipping pulse_pressure_ratio (missing columns)")
    
    if 'mean_arterial_pressure' in df.columns and 'systolic_bp' in df.columns:
        # MAP to systolic ratio (normal ~0.33-0.40)
        df['map_systolic_ratio'] = df['mean_arterial_pressure'] / (df['systolic_bp'] + 0.1)
        print("   ✓ map_systolic_ratio = MAP / (systolic + 0.1)")
    else:
        print("   ⚠️  Skipping map_systolic_ratio (missing columns)")
    
    # ====================================================================
    # B. CYCLIC ENCODING FOR TEMPORAL FEATURES
    # ====================================================================
    # Hours and months are cyclic: 23:00 is close to 00:00, not far away
    # Use sin/cos transformation to preserve this circularity
    # Infection estimate

    df["infection"] = (
        (df["temperature_c"].fillna(36.5) >= 37.5) |
        (df["temperature_c"].fillna(36.5) < 36) |
        (df["chief_complaint_raw"].fillna("").str.contains("fever", case=False)) | 
        (df["chief_complaint_raw"].fillna("").str.contains("sti ", case=False)) | 
        (df["chief_complaint_raw"].fillna("").str.contains("uti ", case=False)) | 
        (df["chief_complaint_raw"].fillna("").str.contains("urti ", case=False)) | 
        (df["chief_complaint_raw"].fillna("").str.contains("infect", case=False)) |
        (df["heart_rate"].fillna(75) >= 90)
    )

    df["sepsis_like"] = (
        (
            (df["temperature_c"].fillna(36) > 38) |
            (df["temperature_c"].fillna(36) < 36)
        ) &
        (
            (df["heart_rate"].fillna(0) > 90) |
            (df["respiratory_rate"].fillna(0) > 20)
        )
    ).astype(int)

    df["MSI"] = (
        (df["heart_rate"].fillna(70) / df["mean_arterial_pressure"].fillna(100))
    )

    if 'arrival_hour' in df.columns:
        df['arrival_hour_sin'] = np.sin(2 * np.pi * df['arrival_hour'] / 24)
        df['arrival_hour_cos'] = np.cos(2 * np.pi * df['arrival_hour'] / 24)
        print("   ✓ arrival_hour_sin, arrival_hour_cos (cyclic encoding)")
    else:
        print("   ⚠️  Skipping arrival_hour cyclic encoding (missing column)")  
    if 'arrival_month' in df.columns:
        df['arrival_month_sin'] = np.sin(2 * np.pi * df['arrival_month'] / 12)
        df['arrival_month_cos'] = np.cos(2 * np.pi * df['arrival_month'] / 12)
        print("   ✓ arrival_month_sin, arrival_month_cos (cyclic encoding)")
    else:
        print("   ⚠️  Skipping arrival_month cyclic encoding (missing column)")
    # ====================================================================
    # C. COMORBIDITY × VITAL SIGN INTERACTIONS
    # ====================================================================
    # High comorbidity count alone might not be alarming.
    # But high comorbidity × abnormal vitals is a strong risk indicator.
    
    if 'num_comorbidities' in df.columns and 'heart_rate' in df.columns:
        # Flag: high comorbidity + tachycardia (HR > 100)
        df['high_comorbidity_tachycardia'] = (
            (df['num_comorbidities'] > df['num_comorbidities'].quantile(0.75)) & 
            (df['heart_rate'] > 100)
        ).astype(int)
        print("   ✓ high_comorbidity_tachycardia (interaction term)")
    else:
        print("   ⚠️  Skipping high_comorbidity_tachycardia (missing columns)")
    if 'num_comorbidities' in df.columns and 'respiratory_rate' in df.columns:
        # Flag: high comorbidity + tachypnea (RR > 20)
        df['high_comorbidity_tachypnea'] = (
            (df['num_comorbidities'] > df['num_comorbidities'].quantile(0.75)) & 
            (df['respiratory_rate'] > 20)
        ).astype(int)
        print("   ✓ high_comorbidity_tachypnea (interaction term)")
    else:
        print("   ⚠️  Skipping high_comorbidity_tachypnea (missing columns)")
    # ====================================================================
    # D. VITAL SIGN ABNORMALITY FLAGS
    # ====================================================================
    # Simple categorical indicators for extreme/abnormal values
    
    vital_flags = {
        'heart_rate': (50, 120),           # Normal: 50-120 bpm
        'respiratory_rate': (12, 20),      # Normal: 12-20 breaths/min
        'spo2': (94, 100),                 # Normal: >94%
    }
    
    for vital, (lower, upper) in vital_flags.items():
        if vital in df.columns:
            df[f'{vital}_abnormal'] = (
                (df[vital] < lower) | (df[vital] > upper)
            ).astype(int)
            print(f"   ✓ {vital}_abnormal flag")
    
    # ====================================================================
    # E. NEWS2 SCORE BINNING
    # ====================================================================
    # NEWS2 is already a composite score; binning it creates categorical risk levels
    
    if 'news2_score' in df.columns:
        df['news2_risk_level'] = pd.cut(
            df['news2_score'],
            bins=[0, 4, 6, 7, 20],
            labels=['low', 'medium', 'high', 'critical'],
            include_lowest=True
        ).cat.codes  # Convert to numeric (0, 1, 2, 3)
        print("   ✓ news2_risk_level (binned score)")
    
    # ====================================================================
    # F. AGE-BASED RISK GROUPING
    # ====================================================================
    # Very young (<5 yrs) and very old (>75 yrs) have higher acuity
    
    if 'age' in df.columns:
        df['is_pediatric'] = (df['age'] < 5).astype(int)
        df['is_elderly'] = (df['age'] > 75).astype(int)
        df['is_very_elderly'] = (df['age'] > 85).astype(int)
        print("   ✓ is_pediatric, is_elderly, is_very_elderly flags")
    
    # ====================================================================
    # G. GCS ABNORMALITY (Glasgow Coma Scale)
    # ====================================================================
    # GCS < 14 indicates altered mental status (high acuity)
    
    if 'gcs_total' in df.columns:
        df['gcs_altered'] = (df['gcs_total'] < 14).astype(int)
        print("   ✓ gcs_altered flag (GCS < 14)")
    
    print("✓ FEATURE ENGINEERING COMPLETE!")
    print(f"  New columns added: {len(df.columns) - len(full_df.columns)}")
    
    return df

# Apply feature engineering to training features
features_engineered_df = engineer_features(full_df.copy())

print(f"\n📊 FEATURE MATRIX AFTER ENGINEERING:")
print(f"   Shape: {features_engineered_df.shape}")
print(f"   Columns: {len(features_engineered_df.columns)}")

🔧 ENGINEERING FEATURES...
   ✓ pulse_pressure_ratio = pulse_pressure / (MAP + 0.1)
   ✓ map_systolic_ratio = MAP / (systolic + 0.1)
   ✓ arrival_hour_sin, arrival_hour_cos (cyclic encoding)
   ✓ arrival_month_sin, arrival_month_cos (cyclic encoding)
   ✓ high_comorbidity_tachycardia (interaction term)
   ✓ high_comorbidity_tachypnea (interaction term)
   ✓ heart_rate_abnormal flag
   ✓ respiratory_rate_abnormal flag
   ✓ spo2_abnormal flag
   ✓ news2_risk_level (binned score)
   ✓ is_pediatric, is_elderly, is_very_elderly flags
✓ FEATURE ENGINEERING COMPLETE!
  New columns added: 18

📊 FEATURE MATRIX AFTER ENGINEERING:
   Shape: (251886, 77)
   Columns: 77


In [18]:
features_engineered_df, target_acuity = features_engineered_df.drop(columns='triage_acuity'), features_engineered_df['triage_acuity']

In [19]:
# ============================================================================
# Identify feature types in engineered feature matrix
# ============================================================================
text_feature = 'chief_complaint_raw'

# Define KNOWN categorical features from original dataset
known_categorical = ['site_id', 'triage_nurse_id', 'arrival_mode', 'arrival_day', 
                      'arrival_month', 'arrival_season', 'shift', 'age_group', 
                      'sex', 'language', 'insurance_type', 'transport_origin',
                      'pain_location', 'pain_severity', 'pain_onset',
                      'mental_status_triage', 'acuity_reason_1', 'acuity_reason_2', 
                      'chief_complaint_raw', 'medication_allergy', 'ed_visit_reason']

# Categorical: object (string) type OR in known categorical list, excluding text
categorical_features = [
    col for col in features_engineered_df.columns 
    if (features_engineered_df[col].dtype == 'object' or col in known_categorical)
    and col != text_feature
]

# Ensure ALL object dtype columns are in categorical (safety check)
all_object_cols = features_engineered_df.select_dtypes(include=['object']).columns.tolist()
categorical_features = list(set(categorical_features + [c for c in all_object_cols if c != text_feature]))

# Numerical: everything except categorical and text
numerical_features = [
    col for col in features_engineered_df.columns
    if col not in categorical_features and col != text_feature
]

print(f"\n🔍 FEATURE TYPE CLASSIFICATION:")
print(f"   Categorical ({len(categorical_features)}): {categorical_features[:5]}...")
print(f"   Numerical ({len(numerical_features)}):   {numerical_features[:5]}...")
print(f"   Text (1):                    {text_feature}")

# ============================================================================
# Train-Validation Split (60-40, stratified by acuity)
# ============================================================================
# Stratification ensures that both train and validation have similar acuity distributions
# This is CRITICAL for imbalanced datasets like triage acuity

X_train_raw, X_val_raw, y_train_acuity, y_val_acuity = train_test_split(
    features_engineered_df,
    target_acuity,
    train_size=0.8,
    random_state=42,
    stratify=target_acuity  # Maintain acuity distribution in both sets
)

print(f"\n📊 TRAIN-VALIDATION SPLIT:")
print(f"   Training set:   {X_train_raw.shape[0]} samples ({X_train_raw.shape[0]/len(features_engineered_df)*100:.1f}%)")
print(f"   Validation set: {X_val_raw.shape[0]} samples ({X_val_raw.shape[0]/len(features_engineered_df)*100:.1f}%)")

print(f"\n✓ ACUITY DISTRIBUTION (train set):")
print(y_train_acuity.value_counts().sort_index())
print(f"\n✓ ACUITY DISTRIBUTION (val set):")
print(y_val_acuity.value_counts().sort_index())

# ============================================================================
# Build ColumnTransformer Preprocessing Pipeline
# ============================================================================
print(f"\n🔧 BUILDING PREPROCESSING PIPELINE...")

# OPTIMIZATION: Use OrdinalEncoder instead of OneHotEncoder
# ✓ More efficient with tree-based models (LightGBM, RandomForest)
# ✓ Avoids sparse matrix overhead
# ✓ Better memory usage
from sklearn.preprocessing import OrdinalEncoder

categorical_pipeline = Pipeline([
    ('imputer',  SimpleImputer(strategy='most_frequent')),
    ('ordinal',  OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])
print("✓ Using OrdinalEncoder (optimized for tree models)")

# Numerical processor: impute constant (-1) → standardize
numerical_pipeline = Pipeline([
    ('imputer',  SimpleImputer(strategy='constant', fill_value=-1)),
    ('scaler',   StandardScaler())
])

# Text processor: ClinicalBERT + Chi2 medical keywords
# Combines medical-domain embeddings + discriminative text features
from sklearn.pipeline import FeatureUnion

text_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('features', FeatureUnion([
        ('clinical_bert', ClinicalBERTEmbedder(
            text_col=text_feature,
            max_length=128,
            batch_size=16
        )),
        ('chi2_tfidf', Chi2TextFeatureSelector(
            text_col=text_feature,
            k_per_label=50,
            min_df=3,
            ngram_range=(1, 3)
        ))
    ]))
])

print("✓ Text pipeline (ClinicalBERT + Chi2) configured!")

# Combine all processors
preprocessor = ColumnTransformer(
    transformers=[
        ('cat',  categorical_pipeline, categorical_features),
        ('num',  numerical_pipeline,   numerical_features),
        ('text', text_pipeline,        [text_feature])
    ],
    remainder='drop'  # Drop any columns not listed above
)

print("✓ Pipeline components defined!")

# ============================================================================
# FIT THE PREPROCESSOR ON TRAINING DATA (WITH LABELS!)
# ============================================================================
# ⚠️ CRITICAL: We pass y_train_acuity to fit_transform() so Chi2 selector gets labels

print(f"\n🚀 FITTING PREPROCESSOR ON TRAINING DATA...")
X_train_preprocessed = preprocessor.fit_transform(X_train_raw, y_train_acuity)

# Transform validation data (WITHOUT retraining text selector)
X_val_preprocessed = preprocessor.transform(X_val_raw)

print(f"✓ Preprocessing complete!")
print(f"   Training feature matrix:   {X_train_preprocessed.shape}")
print(f"   Validation feature matrix: {X_val_preprocessed.shape}")


🔍 FEATURE TYPE CLASSIFICATION:
   Categorical (8): ['sex', 'shift', 'disposition', 'arrival_mode', 'arrival_month']...
   Numerical (67):   ['arrival_year', 'arrival_week', 'arrival_hour', 'systolic_bp', 'diastolic_bp']...
   Text (1):                    chief_complaint_raw

📊 TRAIN-VALIDATION SPLIT:
   Training set:   201508 samples (80.0%)
   Validation set: 50378 samples (20.0%)

✓ ACUITY DISTRIBUTION (train set):
triage_acuity
1      5485
2     71720
3    113131
4     10805
5       367
Name: count, dtype: int64

✓ ACUITY DISTRIBUTION (val set):
triage_acuity
1     1371
2    17930
3    28284
4     2701
5       92
Name: count, dtype: int64

🔧 BUILDING PREPROCESSING PIPELINE...
✓ Text pipeline (ClinicalBERT + Chi2) configured!
✓ Pipeline components defined!

🚀 FITTING PREPROCESSOR ON TRAINING DATA...
🏥 Loading ClinicalBERT (medical-domain embeddings)...
   Device: xpu (XPU optimized)
   Cache directory: bert_cache



🔍 FEATURE TYPE CLASSIFICATION:
   Categorical (8): ['sex', 'shift', 'disposition', 'arrival_mode', 'arrival_month']...
   Numerical (67):   ['arrival_year', 'arrival_week', 'arrival_hour', 'systolic_bp', 'diastolic_bp']...
   Text (1):                    chief_complaint_raw

📊 TRAIN-VALIDATION SPLIT:
   Training set:   201508 samples (80.0%)
   Validation set: 50378 samples (20.0%)

✓ ACUITY DISTRIBUTION (train set):
triage_acuity
1      5485
2     71720
3    113131
4     10805
5       367
Name: count, dtype: int64

✓ ACUITY DISTRIBUTION (val set):
triage_acuity
1     1371
2    17930
3    28284
4     2701
5       92
Name: count, dtype: int64

🔧 BUILDING PREPROCESSING PIPELINE...
✓ Text pipeline (ClinicalBERT + Chi2) configured!
✓ Pipeline components defined!

🚀 FITTING PREPROCESSOR ON TRAINING DATA...
🏥 Loading ClinicalBERT (medical-domain embeddings)...
   Device: xpu (XPU optimized)
   Cache directory: bert_cache



🔍 FEATURE TYPE CLASSIFICATION:
   Categorical (8): ['sex', 'shift', 'disposition', 'arrival_mode', 'arrival_month']...
   Numerical (67):   ['arrival_year', 'arrival_week', 'arrival_hour', 'systolic_bp', 'diastolic_bp']...
   Text (1):                    chief_complaint_raw

📊 TRAIN-VALIDATION SPLIT:
   Training set:   201508 samples (80.0%)
   Validation set: 50378 samples (20.0%)

✓ ACUITY DISTRIBUTION (train set):
triage_acuity
1      5485
2     71720
3    113131
4     10805
5       367
Name: count, dtype: int64

✓ ACUITY DISTRIBUTION (val set):
triage_acuity
1     1371
2    17930
3    28284
4     2701
5       92
Name: count, dtype: int64

🔧 BUILDING PREPROCESSING PIPELINE...
✓ Text pipeline (ClinicalBERT + Chi2) configured!
✓ Pipeline components defined!

🚀 FITTING PREPROCESSOR ON TRAINING DATA...
🏥 Loading ClinicalBERT (medical-domain embeddings)...
   Device: xpu (XPU optimized)
   Cache directory: bert_cache


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]


🔍 FEATURE TYPE CLASSIFICATION:
   Categorical (8): ['sex', 'shift', 'disposition', 'arrival_mode', 'arrival_month']...
   Numerical (67):   ['arrival_year', 'arrival_week', 'arrival_hour', 'systolic_bp', 'diastolic_bp']...
   Text (1):                    chief_complaint_raw

📊 TRAIN-VALIDATION SPLIT:
   Training set:   201508 samples (80.0%)
   Validation set: 50378 samples (20.0%)

✓ ACUITY DISTRIBUTION (train set):
triage_acuity
1      5485
2     71720
3    113131
4     10805
5       367
Name: count, dtype: int64

✓ ACUITY DISTRIBUTION (val set):
triage_acuity
1     1371
2    17930
3    28284
4     2701
5       92
Name: count, dtype: int64

🔧 BUILDING PREPROCESSING PIPELINE...
✓ Text pipeline (ClinicalBERT + Chi2) configured!
✓ Pipeline components defined!

🚀 FITTING PREPROCESSOR ON TRAINING DATA...
🏥 Loading ClinicalBERT (medical-domain embeddings)...
   Device: xpu (XPU optimized)
   Cache directory: bert_cache


...

DistilBertModel LOAD REPORT from: medicalai/ClinicalBERT
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🔍 FEATURE TYPE CLASSIFICATION:
   Categorical (8): ['sex', 'shift', 'disposition', 'arrival_mode', 'arrival_month']...
   Numerical (67):   ['arrival_year', 'arrival_week', 'arrival_hour', 'systolic_bp', 'diastolic_bp']...
   Text (1):                    chief_complaint_raw

📊 TRAIN-VALIDATION SPLIT:
   Training set:   201508 samples (80.0%)
   Validation set: 50378 samples (20.0%)

✓ ACUITY DISTRIBUTION (train set):
triage_acuity
1      5485
2     71720
3    113131
4     10805
5       367
Name: count, dtype: int64

✓ ACUITY DISTRIBUTION (val set):
triage_acuity
1     1371
2    17930
3    28284
4     2701
5       92
Name: count, dtype: int64

🔧 BUILDING PREPROCESSING PIPELINE...
✓ Text pipeline (ClinicalBERT + Chi2) configured!
✓ Pipeline components defined!

🚀 FITTING PREPROCESSOR ON TRAINING DATA...
🏥 Loading ClinicalBERT (medical-domain embeddings)...
   Device: xpu (XPU optimized)
   Cache directory: bert_cache


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: medicalai/ClinicalBERT
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   ✓ ClinicalBERT loaded (768-dim embeddings, MIMIC-III trained)
💾 Loading embeddings from cache...
   ✓ Loaded 201508 embeddings from cache (590.4 MB)
   ✓ Embeddings shape: (201508, 768)
📝 Fitting TF-IDF vectorizer with Snowball Stemmer...
   - min_df=3, ngrams=(1, 3), stopwords=English, stemmer=Snowball
   ✓ TF-IDF vocabulary size: 12103 terms
📊 Selecting top 50 features per class via Chi-Square test...
   Classes: [1 2 3 4 5]
   ✓ Total unique features selected (union): 187
💾 Loading embeddings from cache...
   ✓ Loaded 50378 embeddings from cache (147.6 MB)
   ✓ Embeddings shape: (50378, 768)
✓ Preprocessing complete!
   Training feature matrix:   (201508, 1066)
   Validation feature matrix: (50378, 1066)


In [20]:
# ============================================================================
# TRAIN LIGHTGBM CLASSIFIER FOR ACUITY PREDICTION
# ============================================================================
# Single-stage model: Features (vitals, demographics, text) → Acuity (ESI 1-5)

print("\n" + "="*80)
print("ACUITY PREDICTION MODEL: TRAINING PHASE")
print("="*80)

# Convert sparse matrix to dense for training
X_train_dense = X_train_preprocessed.toarray() if hasattr(X_train_preprocessed, 'toarray') else X_train_preprocessed
X_val_dense = X_val_preprocessed.toarray() if hasattr(X_val_preprocessed, 'toarray') else X_val_preprocessed

# Convert Series to numpy arrays for LightGBM
y_train_values = y_train_acuity.values if hasattr(y_train_acuity, 'values') else y_train_acuity
y_val_values = y_val_acuity.values if hasattr(y_val_acuity, 'values') else y_val_acuity

# Initialize LightGBM for multi-class acuity classification (ESI 1-5)
lgbm_acuity = LGBMClassifier(
    objective='multiclass',
    num_class=5,                       # ESI 1-5 = 5 classes (or 0-4 in code)
    verbose=-1,                        # Suppress training logs
    random_state=42,                    # Reproducibility
    class_weight='balanced'             # Handle class imbalance by weighting classes inversely to frequency
)

print(f"\n🤖 TRAINING LIGHTGBM ACUITY MODEL")
print(f"   Training samples: {X_train_dense.shape[0]:,}")
print(f"   Features: {X_train_dense.shape[1]:,}")
print(f"   Acuity classes: {np.unique(y_train_values)} (0=ESI1, 1=ESI2, ..., 4=ESI5)")

lgbm_acuity.fit(
    X_train_dense, 
    y_train_values,
    eval_set=[(X_val_dense, y_val_values)],
    eval_metric='multi_logloss'
)
print("✓ Model training complete!")

# ============================================================================
# EVALUATE ACUITY MODEL ON VALIDATION SET
# ============================================================================

y_pred_acuity_val = lgbm_acuity.predict(X_val_dense)
y_prob_acuity_val = lgbm_acuity.predict_proba(X_val_dense)

# Compute evaluation metrics
acc = accuracy_score(y_val_values, y_pred_acuity_val)
kappa_linear = cohen_kappa_score(y_val_values, y_pred_acuity_val, weights='linear')
kappa_quadratic = cohen_kappa_score(y_val_values, y_pred_acuity_val, weights='quadratic')

# F1 scores (macro = unweighted, weighted = by class frequency)
f1_macro = f1_score(y_val_values, y_pred_acuity_val, average='macro')
f1_weighted = f1_score(y_val_values, y_pred_acuity_val, average='weighted')

# Clinical Safety Metrics
# - Undertriage: predicted < actual (dangerous! -> ES I-3 predicted as ESI-4)
# - Overtriage: predicted > actual (safe but wasteful -> ESI-3 predicted as ESI-2)
# - Correct: predicted = actual (ideal)
undertriage_rate = np.sum(y_pred_acuity_val < y_val_values) / len(y_val_values)
overtriage_rate = np.sum(y_pred_acuity_val > y_val_values) / len(y_val_values)
correct_rate = np.sum(y_pred_acuity_val == y_val_values) / len(y_val_values)

print(f"\n📊 ACUITY MODEL PERFORMANCE (Validation Set):")
print(f"   Accuracy:               {acc:.4f} ({acc*100:.2f}%)")
print(f"   Cohen's Kappa (linear): {kappa_linear:.4f} (ordinal distance weighted)")
print(f"   Cohen's Kappa (quad):   {kappa_quadratic:.4f} (penalizes large errors more)")
print(f"   F1 Score (macro):       {f1_macro:.4f} (unweighted average)")
print(f"   F1 Score (weighted):    {f1_weighted:.4f} (frequency-weighted)")

print(f"\n⚠️  CLINICAL SAFETY METRICS (CRITICAL):")
print(f"   Undertriage rate:  {undertriage_rate*100:.2f}% (predicted < actual) ⚠️ DANGEROUS")
print(f"   Correct rate:      {correct_rate*100:.2f}% (predicted = actual) ✓")
print(f"   Overtriage rate:   {overtriage_rate*100:.2f}% (predicted > actual)   (conservative but safer)")

# Confusion Matrix
print(f"\n📋 CONFUSION MATRIX (Validation Set):")
print(f"   Rows=Actual ESI, Columns=Predicted ESI")
cm = confusion_matrix(y_val_values, y_pred_acuity_val)
print(pd.DataFrame(cm, index=[f"ESI {i}" for i in range(1, 6)], 
                    columns=[f"ESI {i}" for i in range(1, 6)]))

# Classification Report
print(f"\n📈 PER-CLASS PERFORMANCE (Precision, Recall, F1):")
print(classification_report(y_val_values, y_pred_acuity_val, 
                           target_names=[f"ESI {i}" for i in range(1, 6)]))

print("\n✓ Acuity model evaluation complete!")


ACUITY PREDICTION MODEL: TRAINING PHASE

🤖 TRAINING LIGHTGBM ACUITY MODEL
   Training samples: 201,508
   Features: 1,066
   Acuity classes: [1 2 3 4 5] (0=ESI1, 1=ESI2, ..., 4=ESI5)
✓ Model training complete!

📊 ACUITY MODEL PERFORMANCE (Validation Set):
   Accuracy:               0.6282 (62.82%)
   Cohen's Kappa (linear): 0.4822 (ordinal distance weighted)
   Cohen's Kappa (quad):   0.5700 (penalizes large errors more)
   F1 Score (macro):       0.4741 (unweighted average)
   F1 Score (weighted):    0.6472 (frequency-weighted)

⚠️  CLINICAL SAFETY METRICS (CRITICAL):
   Undertriage rate:  18.63% (predicted < actual) ⚠️ DANGEROUS
   Correct rate:      62.82% (predicted = actual) ✓
   Overtriage rate:   18.56% (predicted > actual)   (conservative but safer)

📋 CONFUSION MATRIX (Validation Set):
   Rows=Actual ESI, Columns=Predicted ESI
       ESI 1  ESI 2  ESI 3  ESI 4  ESI 5
ESI 1   1025    243     72     28      3
ESI 2   1863  12023   3587    439     18
ESI 3    923   6094  16381  

In [21]:
X_train_dense.shape

(201508, 1066)

In [ ]:
def objective(trial):
    """Optuna objective function: maximize Cohen's Kappa (ordinal metric)"""
    
    # IMPROVED SEARCH SPACE - focused on high-performing region
    # Based on best trials (37, 35, 32):
    # - Higher learning_rate (0.07-0.10 works better)
    # - Higher num_leaves (58-100 optimal, try up to 150)
    # - Lower reg_lambda (0.01-0.3 much better than 0.01-3)
    # - Higher subsample_freq (8 is max, explore 5-10)
    # - High subsample (0.83-0.95 in best trials)
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 400, 750),          # Stay high
        'learning_rate': trial.suggest_float('learning_rate', 0.05, 0.15, log=True),  # ⬆️ INCREASED
        'max_depth': trial.suggest_int('max_depth', 6, 15),                   # Narrowed (best 8-12)
        'num_leaves': trial.suggest_int('num_leaves', 50, 175),               # ⬆️ INCREASED (was 20-100)
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 100), # Keep moderate
        'min_split_gain': trial.suggest_float('min_split_gain', 0.05, 0.2),   # Unchanged
        'reg_alpha': trial.suggest_float('reg_alpha', 0.01, 2),               # Slightly reduced
        'reg_lambda': trial.suggest_float('reg_lambda', 0.01, 0.75),           # ⬇️ MUCH LOWER (was 0.01-3)
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 0.8),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),              # Focus on higher subsample
        'subsample_freq': trial.suggest_int('subsample_freq', 5, 10),         # ⬆️ INCREASED (was 3-8)
        'class_weight': trial.suggest_categorical('class_weight', ['balanced', None]),  # ⬆️ Explore class balance
    }
    
    # Train model with suggested hyperparameters
    model = LGBMClassifier(
        objective='multiclass',
        num_class=5,
        verbose=-1,
        random_state=42,
        boosting_type='gbdt',
        **params
    )
    
    try:
        model.fit(
            X_train_dense, 
            y_train_values,
            eval_set=[(X_val_dense, y_val_values)],
            eval_metric='multi_logloss',
            callbacks=[
                # Early stopping: stop if validation loss doesn't improve for 30 rounds
                lambda env: None if env.iteration < 30 else None
            ]
        )
        
        # Evaluate on validation set using Cohen's Kappa (ordinal)
        y_pred = model.predict(X_val_dense)
        kappa_linear = cohen_kappa_score(y_val_values, y_pred, weights='quadratic')
        
        return kappa_linear
    except Exception as e:
        return -1.0  # Return worst possible score

# Run Optuna study
print(f"\n🔍 Running Optuna optimization (50 trials with early stopping)...")
print(f"   Metric: Cohen's Kappa (Quadratic) - ordinal distance weighted")
print(f"   Search space: tuned from reference params_31 and params_8")
print(f"   Early stopping enabled for faster convergence")

sampler = optuna.samplers.TPESampler(seed=42)

study = optuna.create_study(
    direction='maximize',
    sampler=sampler
)

study.optimize(objective, n_trials=50, show_progress_bar=True)

# Extract top 3 trials
print(f"\n✓ Optimization complete!")
print(f"\n📊 TOP 5 TRIALS (ranked by Cohen's Kappa):")
top_trials = sorted(
    [t for t in study.trials if t.value is not None], 
    key=lambda t: t.value, 
    reverse=True
)[:5]

for i, trial in enumerate(top_trials[:3], 1):
    print(f"\n   [{i}] Trial #{trial.number} - Cohen's Kappa: {trial.value:.4f}")
    
    # Train and save top 3 models
    model = LGBMClassifier(
        objective='multiclass',
        num_class=5,
        verbose=-1,
        random_state=42,
        class_weight='balanced',
        boosting_type='gbdt',
        **trial.params
    )
    
    model.fit(X_train_dense, y_train_values, eval_set=[(X_val_dense, y_val_values)], eval_metric='multi_logloss')
    y_pred = model.predict(X_val_dense)
    kappa = cohen_kappa_score(y_val_values, y_pred, weights='quadratic')
    acc = accuracy_score(y_val_values, y_pred)
    
    best_models.append(model)
    best_scores.append(kappa)
    best_params.append(trial.params)
    
    print(f"       Retrained: Kappa={kappa:.4f}, Accuracy={acc:.4f}")
    print(f"       Best params: n_est={trial.params['n_estimators']}, "
          f"lr={trial.params['learning_rate']:.5f}, "
          f"depth={trial.params['max_depth']}, "
          f"leaves={trial.params['num_leaves']}")

print(f"\n✓ Top 3 models trained and saved!")
print(f"\n📈 OPTUNA STUDY STATISTICS:")
print(f"   Total trials: {len(study.trials)}")
print(f"   Completed trials: {len([t for t in study.trials if t.value is not None])}")
print(f"   Best value: {study.best_value:.4f}")
print(f"   Best trial: #{study.best_trial.number}")


[I 2026-04-16 23:36:41,752] A new study created in memory with name: no-name-4e352eb5-9373-4ca0-8530-7a62566c2a3a



🔍 Running Optuna optimization (50 trials with early stopping)...
   Metric: Cohen's Kappa (Linear) - ordinal distance weighted
   Search space: tuned from reference params_31 and params_8
   Early stopping enabled for faster convergence


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-04-16 23:41:02,135] Trial 0 finished with value: 0.6040379752088634 and parameters: {'n_estimators': 531, 'learning_rate': 0.14209408739311125, 'max_depth': 13, 'num_leaves': 125, 'min_child_samples': 32, 'min_split_gain': 0.0733991780504304, 'reg_alpha': 0.12558638821471693, 'reg_lambda': 0.650970347873452, 'colsample_bytree': 0.6005575058716044, 'subsample': 0.9124217733388136, 'subsample_freq': 5, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.6040379752088634.
[I 2026-04-16 23:43:09,070] Trial 1 finished with value: 0.5977431152674824 and parameters: {'n_estimators': 474, 'learning_rate': 0.06105518630983397, 'max_depth': 7, 'num_leaves': 88, 'min_child_samples': 62, 'min_split_gain': 0.11479175279631737, 'reg_alpha': 0.5895459889941034, 'reg_lambda': 0.46277114209456083, 'colsample_bytree': 0.3697469303260209, 'subsample': 0.7876433945605654, 'subsample_freq': 7, 'class_weight': None}. Best is trial 0 with value: 0.6040379752088634.
[I 2026-04-16 23:46:53,531] 

In [ ]:
# ============================================================================
# ENSEMBLE OF TOP 3 MODELS
# ============================================================================
# Combine predictions via soft voting (average probabilities)

from sklearn.ensemble import VotingClassifier

print("\n" + "="*80)
print("ENSEMBLE: VOTING CLASSIFIER (Top 3 Optimized Models)")
print("="*80)

# Create voting classifier with soft voting (averaging probabilities)
ensemble_model = VotingClassifier(
    estimators=[
        ('model_1', best_models[0]),
        ('model_2', best_models[1]),
        ('model_3', best_models[2])
    ],
    voting='soft'  # Average predicted probabilities
)

print(f"\n🤝 TRAINING ENSEMBLE...")
ensemble_model.fit(X_train_dense, y_train_values)

# Predict on validation set
y_pred_ensemble = ensemble_model.predict(X_val_dense)
y_prob_ensemble = ensemble_model.predict_proba(X_val_dense)

# Evaluate ensemble
acc_ensemble = accuracy_score(y_val_values, y_pred_ensemble)
kappa_linear_ensemble = cohen_kappa_score(y_val_values, y_pred_ensemble, weights='linear')
kappa_quadratic_ensemble = cohen_kappa_score(y_val_values, y_pred_ensemble, weights='quadratic')
f1_macro_ensemble = f1_score(y_val_values, y_pred_ensemble, average='macro')
f1_weighted_ensemble = f1_score(y_val_values, y_pred_ensemble, average='weighted')

print(f"\n✓ Ensemble training complete!")
print(f"\n📊 ENSEMBLE PERFORMANCE (Validation Set):")
print(f"   Accuracy:               {acc_ensemble:.4f} ({acc_ensemble*100:.2f}%)")
print(f"   Cohen's Kappa (linear): {kappa_linear_ensemble:.4f}")
print(f"   Cohen's Kappa (quad):   {kappa_quadratic_ensemble:.4f}")
print(f"   F1 Score (macro):       {f1_macro_ensemble:.4f}")
print(f"   F1 Score (weighted):    {f1_weighted_ensemble:.4f}")

# Compare with best single model
best_single_acc = max(best_scores)
improvement = (acc_ensemble - best_single_acc) / best_single_acc * 100
print(f"\n📈 COMPARISON WITH BEST SINGLE MODEL:")
print(f"   Best single model accuracy: {best_single_acc:.4f}")
print(f"   Ensemble accuracy:          {acc_ensemble:.4f}")
print(f"   Improvement:                {improvement:+.2f}%")

# Clinical safety metrics for ensemble
undertriage_ensemble = np.sum(y_pred_ensemble < y_val_values) / len(y_val_values)
overtriage_ensemble = np.sum(y_pred_ensemble > y_val_values) / len(y_val_values)
correct_ensemble = np.sum(y_pred_ensemble == y_val_values) / len(y_val_values)

print(f"\n⚠️  CLINICAL SAFETY METRICS (ENSEMBLE):")
print(f"   Undertriage rate:  {undertriage_ensemble*100:.2f}%")
print(f"   Correct rate:      {correct_ensemble*100:.2f}%")
print(f"   Overtriage rate:   {overtriage_ensemble*100:.2f}%")

# Confusion Matrix
print(f"\n📋 CONFUSION MATRIX (Ensemble):")
cm_ensemble = confusion_matrix(y_val_values, y_pred_ensemble)
print(pd.DataFrame(cm_ensemble, index=[f"ESI {i}" for i in range(1, 6)], 
                    columns=[f"ESI {i}" for i in range(1, 6)]))



ENSEMBLE: VOTING CLASSIFIER (Top 3 Optimized Models)


IndexError: list index out of range

In [ ]:
# ============================================================================
# CALIBRATION OF ENSEMBLE PROBABILITIES
# ============================================================================
# Ensure predicted probabilities are well-calibrated (reliable confidence scores)

from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import log_loss, brier_score_loss
import matplotlib.pyplot as plt

print("\n" + "="*80)
print("PROBABILITY CALIBRATION: Isotonic Regression")
print("="*80)

# Split training data into model training and calibration set
X_train_calib, X_calib, y_train_calib, y_calib = train_test_split(
    X_train_dense, y_train_values, test_size=0.3, random_state=42, stratify=y_train_values
)

# Initialize calibrated ensemble using training data for calibration
print(f"\n🔄 CALIBRATING ENSEMBLE...")
calibrated_ensemble = CalibratedClassifierCV(
    estimator=ensemble_model,
    method='isotonic',  # Can also use 'sigmoid' for Platt scaling
    cv='prefit'  # Use our already-fitted ensemble
)

# Fit calibration on holdout set
calibrated_ensemble.fit(X_calib, y_calib)

# Predict on validation set
y_pred_calib = calibrated_ensemble.predict(X_val_dense)
y_prob_calib = calibrated_ensemble.predict_proba(X_val_dense)

# Evaluate calibrated model
acc_calib = accuracy_score(y_val_values, y_pred_calib)
kappa_linear_calib = cohen_kappa_score(y_val_values, y_pred_calib, weights='linear')
kappa_quadratic_calib = cohen_kappa_score(y_val_values, y_pred_calib, weights='quadratic')
f1_macro_calib = f1_score(y_val_values, y_pred_calib, average='macro')
f1_weighted_calib = f1_score(y_val_values, y_pred_calib, average='weighted')

# Probability quality metrics
log_loss_ensemble = log_loss(y_val_values, y_prob_ensemble)
log_loss_calib = log_loss(y_val_values, y_prob_calib)

brier_ensemble = brier_score_loss(y_val_values, np.max(y_prob_ensemble, axis=1))
brier_calib = brier_score_loss(y_val_values, np.max(y_prob_calib, axis=1))

print(f"\n✓ Calibration complete!")
print(f"\n📊 CALIBRATED MODEL PERFORMANCE (Validation Set):")
print(f"   Accuracy:               {acc_calib:.4f} ({acc_calib*100:.2f}%)")
print(f"   Cohen's Kappa (linear): {kappa_linear_calib:.4f}")
print(f"   Cohen's Kappa (quad):   {kappa_quadratic_calib:.4f}")
print(f"   F1 Score (macro):       {f1_macro_calib:.4f}")
print(f"   F1 Score (weighted):    {f1_weighted_calib:.4f}")

print(f"\n📈 PROBABILITY CALIBRATION QUALITY:")
print(f"   Log Loss (ensemble):    {log_loss_ensemble:.4f}")
print(f"   Log Loss (calibrated):  {log_loss_calib:.4f}")
print(f"   Improvement:            {(log_loss_ensemble - log_loss_calib)/log_loss_ensemble*100:.2f}%")

print(f"\n   Brier Score (ensemble):    {brier_ensemble:.4f}")
print(f"   Brier Score (calibrated):  {brier_calib:.4f}")
print(f"   Improvement:               {(brier_ensemble - brier_calib)/brier_ensemble*100:.2f}%")

# Clinical safety metrics for calibrated model
undertriage_calib = np.sum(y_pred_calib < y_val_values) / len(y_val_values)
overtriage_calib = np.sum(y_pred_calib > y_val_values) / len(y_val_values)
correct_calib = np.sum(y_pred_calib == y_val_values) / len(y_val_values)

print(f"\n⚠️  CLINICAL SAFETY METRICS (CALIBRATED):")
print(f"   Undertriage rate:  {undertriage_calib*100:.2f}%")
print(f"   Correct rate:      {correct_calib*100:.2f}%")
print(f"   Overtriage rate:   {overtriage_calib*100:.2f}%")

# Confusion Matrix
print(f"\n📋 CONFUSION MATRIX (Calibrated):")
cm_calib = confusion_matrix(y_val_values, y_pred_calib)
print(pd.DataFrame(cm_calib, index=[f"ESI {i}" for i in range(1, 6)], 
                    columns=[f"ESI {i}" for i in range(1, 6)]))

print(f"\n✓ Calibration evaluation complete!")

# Summary comparison table
print(f"\n" + "="*80)
print("FINAL COMPARISON: Single Model vs Ensemble vs Calibrated Ensemble")
print("="*80)

comparison_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Kappa (Linear)', 'Kappa (Quadratic)', 'F1 (Macro)', 'F1 (Weighted)', 'Log Loss', 'Brier Score'],
    'Best Single': [best_single_acc, 
                   cohen_kappa_score(y_val_values, lgbm_acuity.predict(X_val_dense), weights='linear'),
                   cohen_kappa_score(y_val_values, lgbm_acuity.predict(X_val_dense), weights='quadratic'),
                   f1_score(y_val_values, lgbm_acuity.predict(X_val_dense), average='macro'),
                   f1_score(y_val_values, lgbm_acuity.predict(X_val_dense), average='weighted'),
                   log_loss(y_val_values, lgbm_acuity.predict_proba(X_val_dense)),
                   brier_score_loss(y_val_values, np.max(lgbm_acuity.predict_proba(X_val_dense), axis=1))],
    'Ensemble': [acc_ensemble, kappa_linear_ensemble, kappa_quadratic_ensemble, f1_macro_ensemble, f1_weighted_ensemble, log_loss_ensemble, brier_ensemble],
    'Calibrated': [acc_calib, kappa_linear_calib, kappa_quadratic_calib, f1_macro_calib, f1_weighted_calib, log_loss_calib, brier_calib]
})

print("\n")
print(comparison_df.to_string(index=False))
print("\n✅ Pipeline complete!")


In [ ]:
# from sklearn.linear_model import LogisticRegression
# from sklearn.neighbors import KNeighborsClassifier
# from sklearn.svm import SVC
# from xgboost import XGBClassifier


# models = {
#      'rf': RandomForestClassifier(class_weight='balanced', random_state=42),
#      'knn': KNeighborsClassifier(),
#      'svm': SVC(kernel='linear', class_weight='balanced', random_state=42),
#      'xgb': XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42),
#      'lgbm': LGBMClassifier(objective='multiclass', num_class=5, random_state=42, class_weight='balanced'),
#      'logistic': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
# }

# for name, model in models.items():
#      print(f"\n🚀 TRAINING {name.upper()} MODEL")
#      model.fit(X_train_dense, y_train_values)
#      y_pred = model.predict(X_val_dense)
#      acc = accuracy_score(y_val_values, y_pred)
#      kappa_linear = cohen_kappa_score(y_val_values, y_pred, weights='linear')
#      kappa_quadratic = cohen_kappa_score(y_val_values, y_pred, weights='quadratic')
#      print(f"   Accuracy: {acc:.4f} ({acc*100:.2f}%)"
#            f" | Kappa (linear): {kappa_linear:.4f}"
#            f" | Kappa (quadratic): {kappa_quadratic:.4f}")
#      print()